# 9c. Fitting origin-destination flows to the junction counts

Rounds 1-4 found settings that carry the right *total* traffic at roughly the surveyed speeds, but not the right *spatial pattern*: Kalimati stays too high and the Baneshwor / Thapathali junctions too low, whatever the global setting. This notebook adjusts the zone-to-zone flows themselves.

**Method (path-based multiplicative OD estimation):** simulate 3 seeds; at each of the 6 tuning junctions take r = counted / modelled entering volume (all nodes within 100 m); route every OD pair on the shortest path over the simulated travel times; multiply its demand by r^0.5 for every tuning junction its path enters (at most x1.5 or /1.5 per iteration); repeat up to 6 times or until all 6 pass GEH < 5.

**Start:** trips loaded at mid-block nodes (never inside a junction), outer zones on main roads, at least 3 main-road lanes, peak factor 0.13.

**Caveat:** this fits the 6 tuning junctions directly, so afterwards they are calibration targets, not validation. Only the 3 held-out junctions (Balaju, Kalanki, Koteshwor), never used here, can validate the result.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("scripts/fit_od_counts.py")

In [ ]:
!pip install -q uxsim==1.14.2

In [ ]:
run("scripts/build_minor_ways.py")
run("scripts/build_od_jica.py", "--data", "data/jica_net")

In [ ]:
run("scripts/fit_od_counts.py", "--start", "--midblock --outer-main --k-peak 0.13", "--run", "--min-main-lanes 3", "--iterations", "6", "--seeds", "3", "--jobs", "3", "--radius", "100", tries=3)

In [ ]:
import pandas as pd
print(pd.read_csv("results/odme/log.csv").to_string(index=False))